# IA-SPA: Notebook 3 -- Florence: Exclusionary Zones & Incremental Deployment

**Interference-Aware Submodular Placement Algorithm**  
Taus, Tsai, Andrews (2026)

---

This notebook reproduces the two advanced Florence experiments from Section IV-E and IV-F:

1. **Exclusionary zone** -- The Basilica di San Lorenzo is declared off-limits for
   transmitter placement.  We show that IA-SPA still achieves a 55% edge-rate
   improvement vs. the Iliad baseline despite this constraint.

2. **Incremental deployment** -- Three pre-existing Iliad towers are fixed as
   T_fixed, and IA-SPA optimally augments the network from this warm start.

---

## Imports and Configuration

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from sionna.rt import load_scene, scene as sionna_scenes, PlanarArray

from ia_spa import (
    TowerOptimizer, run_greedy,
    build_terrain_tree, apply_tower_height,
    evaluate_and_save, load_greedy_positions,
)

with open('../config/config.yaml') as fh:
    cfg = yaml.safe_load(fh)

BASIS_FOLDER   = Path(cfg['paths']['basis_functions_fl'])
TOWER_DATA     = Path(cfg['paths']['tower_data'])
TOWER_HEIGHT   = float(cfg['transmitter']['tower_height_m'])
POWER_DBM      = float(cfg['transmitter']['power_dbm'])
FREQ_HZ        = float(cfg['scene']['frequency_hz'])
HEIGHTS        = cfg['receiver']['heights_m']
BW_HZ          = float(cfg['link']['bandwidth_hz'])
GAMMA          = float(cfg['link']['snr_gap_gamma'])
N_ITER         = int(cfg['optimizer']['n_iter'])
print('Configuration loaded.')

## Part A -- Exclusionary Zone

The exclusionary zone is implemented by **zeroing out basis-function values**
for any candidate inside the ellipse.  This is equivalent to setting
$P(y, x) \equiv 0$ for $x \notin \mathcal{X}$ as described in Section II-C.

We define the ellipse by two foci and a major-axis length in scene coordinates.

In [ ]:
# Ellipse parameters for the Basilica di San Lorenzo exclusion zone
# (adjust to match scene coordinates if needed)
FOCUS_1          = (-30.0, 15.0)   # (x, y) in scene metres
FOCUS_2          = (30.0, 15.0)
MAJOR_AXIS_LEN   = 120.0            # sum-of-distances threshold

def is_inside_ellipse(point_xy, focus1, focus2, major_axis_len):
    d1 = math.dist(point_xy, focus1)
    d2 = math.dist(point_xy, focus2)
    return (d1 + d2) <= major_axis_len

# Load baseline optimizer and check which candidates fall in the exclusion zone
optimizer_base = TowerOptimizer(BASIS_FOLDER, aggregation='max')
in_zone = np.array(
    [is_inside_ellipse(c[:2], FOCUS_1, FOCUS_2, MAJOR_AXIS_LEN)
     for c in optimizer_base.coords]
)
print(f'Candidates inside exclusion zone: {in_zone.sum()} / {optimizer_base.n_candidates}')

In [ ]:
# Create a masked optimizer: zero out basis-function gains for excluded candidates
# by wrapping marginal_gains
class MaskedTowerOptimizer(TowerOptimizer):
    """TowerOptimizer that forbids placement inside an exclusionary zone."""

    def __init__(self, *args, exclusion_mask: np.ndarray, **kwargs):
        super().__init__(*args, **kwargs)
        self.exclusion_mask = exclusion_mask  # True -> forbidden

    def marginal_gains(self, u, density=None):
        gains = super().marginal_gains(u, density)
        gains[self.exclusion_mask] = 0.0
        return gains

optimizer_excl = MaskedTowerOptimizer(
    BASIS_FOLDER, aggregation='max', exclusion_mask=in_zone
)

RESULTS_EXCL = Path(cfg['paths']['results']) / 'FL_exclusion'
run_greedy(optimizer_excl, results_folder=RESULTS_EXCL, n_iter=N_ITER)
print('Exclusionary-zone run complete.')

## Part B -- Incremental Deployment

Here we fix three Iliad towers as T_fixed and ask IA-SPA to optimally
augment the network.  The fixed towers are warm-started into the weight
vector `u` before the greedy loop begins.

In [ ]:
# Select three representative Iliad towers as the fixed baseline
FIXED_INDICES = [1, 5, 8]
iliad_towers  = np.load(TOWER_DATA / 'FL_Iliad.npy')
fixed_towers  = iliad_towers[FIXED_INDICES, :]

print(f'Fixed transmitters (T_fixed): {len(fixed_towers)}')
for i, (x, y, z) in enumerate(fixed_towers):
    print(f'  {i}: ({x:.1f}, {y:.1f}, {z:.1f})')

optimizer_incr = TowerOptimizer(BASIS_FOLDER, aggregation='max')
RESULTS_INCR   = Path(cfg['paths']['results']) / 'FL_incremental'

run_greedy(
    optimizer_incr,
    results_folder=RESULTS_INCR,
    n_iter=N_ITER,
    fixed_towers=fixed_towers,
)
print('Incremental deployment run complete.')

## Evaluate and Compare Both Scenarios

In [ ]:
fl_scene = load_scene(sionna_scenes.florence, merge_shapes=False)
fl_scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern='iso', polarization='V')
fl_scene.frequency = FREQ_HZ
terrain_tree, terrain_verts = build_terrain_tree(fl_scene, 'ground')

iliad_pos = apply_tower_height(iliad_towers, terrain_tree, terrain_verts, TOWER_HEIGHT)
n_iliad   = len(iliad_pos)

# Reference Iliad
save_dir = Path(cfg['paths']['results']) / 'FL_Processed'
evaluate_and_save(fl_scene, 'Iliad_ref', iliad_pos, save_dir, HEIGHTS, POWER_DBM, BW_HZ, GAMMA)

# Exclusionary zone result
greedy_excl = apply_tower_height(
    load_greedy_positions(RESULTS_EXCL), terrain_tree, terrain_verts, TOWER_HEIGHT
)
evaluate_and_save(
    fl_scene, 'Iliad_Greedy_Exclusion',
    greedy_excl[:n_iliad], save_dir,
    HEIGHTS, POWER_DBM, BW_HZ, GAMMA,
)

# Incremental deployment result
greedy_incr = apply_tower_height(
    load_greedy_positions(RESULTS_INCR), terrain_tree, terrain_verts, TOWER_HEIGHT
)
evaluate_and_save(
    fl_scene, 'Iliad_Greedy_Incremental',
    greedy_incr[:n_iliad], save_dir,
    HEIGHTS, POWER_DBM, BW_HZ, GAMMA,
)
print('Evaluation complete.')

In [ ]:
import pandas as pd

def stats(tag):
    r = np.load(save_dir / f'{tag}_rate.npy').ravel() / 1e6
    return {
        'Mean (Mbps)': round(r.mean(), 2),
        '5th pct (Mbps)': round(np.percentile(r, 5), 2),
        'Max (Mbps)': round(r.max(), 2),
    }

table = pd.DataFrame({
    'Iliad (ref)':      stats('Iliad_ref'),
    'IA-SPA + Exclusion': stats('Iliad_Greedy_Exclusion'),
    'IA-SPA + Incremental': stats('Iliad_Greedy_Incremental'),
}).T
print(table.to_string())
table.to_csv(save_dir / 'florence_advanced_table.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
tags   = ['Iliad_ref', 'Iliad_Greedy_Exclusion', 'Iliad_Greedy_Incremental']
titles = ['Iliad Reference', 'IA-SPA (Exclusion Zone)', 'IA-SPA (Incremental)']

maps = [np.load(save_dir / f'{t}_rate.npy') / 1e6 for t in tags]
vmax = max(m.max() for m in maps)

for ax, rate_map, title in zip(axes, maps, titles):
    im = ax.imshow(rate_map, cmap='plasma', vmin=0, vmax=vmax, origin='lower')
    ax.set_title(f'{title}\n{rate_map.mean():.1f} Mbps mean  |  {np.percentile(rate_map, 5):.1f} Mbps edge')
    ax.axis('off')

fig.colorbar(im, ax=axes.tolist(), shrink=0.8, label='Rate (Mbps)')
fig.suptitle('Florence: Reference vs. IA-SPA (Advanced Scenarios)', fontsize=13, weight='bold')
plt.tight_layout()
plt.savefig(save_dir / 'florence_advanced.png', dpi=150, bbox_inches='tight')
plt.show()